In [1]:
import tensorflow as tf
from tensorflow.keras.models import Model
import pandas as pd
import numpy as np
import os
import time
from scipy import stats
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn import tree
from sklearn.model_selection import cross_val_score
from sklearn import metrics
from tensorflow.keras.layers import Input, Conv1D, Dropout, MaxPooling1D, Flatten, Dense, LSTM
from tensorflow.keras.optimizers import Adam
import h5py
from scapy.all import *
from sklearn.model_selection import train_test_split

D:\Anaconda\anaaconda\lib\site-packages\scapy\layers\ipsec.py:469: CryptographyDeprecationWarning: Blowfish has been deprecated
  cipher=algorithms.Blowfish,
D:\Anaconda\anaaconda\lib\site-packages\scapy\layers\ipsec.py:483: CryptographyDeprecationWarning: CAST5 has been deprecated
  cipher=algorithms.CAST5,


In [2]:
df1=pd.read_csv("Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
df2=pd.read_csv("Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv")
df3=pd.read_csv("Friday-WorkingHours-Morning.pcap_ISCX.csv")
df4=pd.read_csv("Monday-WorkingHours.pcap_ISCX.csv")
df5=pd.read_csv("Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv")
df6=pd.read_csv("Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv")
df7=pd.read_csv("Tuesday-WorkingHours.pcap_ISCX.csv")
df8=pd.read_csv("Wednesday-workingHours.pcap_ISCX.csv")

df = pd.concat([df1,df2,df3,df4,df5,df6,df7,df8])

In [3]:
df[' Label'].value_counts()

BENIGN                        2239619
DoS Hulk                       231073
PortScan                       158930
DDoS                           123775
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name:  Label, dtype: int64

In [4]:
df.duplicated().sum()

304658

In [5]:
df =  df.drop_duplicates(keep="first")

In [6]:
df.duplicated().sum()

0

In [7]:
df.dropna(inplace=True)

In [8]:
df=df.groupby(' Label').filter(lambda x:len(x)>10000)
df[' Label'].value_counts()

BENIGN           2066379
DoS Hulk          172846
DDoS              123765
PortScan           90819
DoS GoldenEye      10286
Name:  Label, dtype: int64

In [9]:
integer = []
f = []

for i in df.columns[:-1]:
    if df[i].dtype == "int64": 
        integer.append(i)
    else : 
        f.append(i)

df[integer] = df[integer].astype("int32")
df[f] = df[f].astype("float32")

In [10]:
df = df[~df.isin([np.nan, np.inf, -np.inf]).any(1)]

In [11]:
def correlation(dataset, threshold):
    col_corr = set()  
    corr_matrix = dataset.corr()
    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if abs(corr_matrix.iloc[i, j]) > threshold: 
              colname = corr_matrix.columns[i]                  
              col_corr.add(colname)
    return col_corr

In [12]:
df.shape

(2462901, 79)

In [13]:
corr_features = correlation(df, 0.85)
corr_features

{' Active Min',
 ' Average Packet Size',
 ' Avg Bwd Segment Size',
 ' Avg Fwd Segment Size',
 ' Bwd IAT Max',
 ' Bwd IAT Mean',
 ' Bwd IAT Min',
 ' Bwd Packet Length Mean',
 ' Bwd Packet Length Std',
 ' CWE Flag Count',
 ' ECE Flag Count',
 ' Flow IAT Max',
 ' Fwd Header Length.1',
 ' Fwd IAT Max',
 ' Fwd IAT Mean',
 ' Fwd IAT Min',
 ' Fwd IAT Std',
 ' Fwd Packet Length Mean',
 ' Fwd Packet Length Std',
 ' Idle Max',
 ' Idle Min',
 ' Max Packet Length',
 ' Packet Length Mean',
 ' Packet Length Std',
 ' Packet Length Variance',
 ' SYN Flag Count',
 ' Subflow Bwd Bytes',
 ' Subflow Bwd Packets',
 ' Subflow Fwd Bytes',
 ' Total Backward Packets',
 ' Total Length of Bwd Packets',
 ' act_data_pkt_fwd',
 ' min_seg_size_forward',
 'Fwd IAT Total',
 'Fwd Packets/s',
 'Idle Mean',
 'Subflow Fwd Packets'}

In [14]:
df.drop(corr_features,axis=1,inplace=True)

In [15]:
X = df.drop([' Label'],axis=1)
y = df[' Label']

In [16]:
X.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Bwd Packet Length Max,Bwd Packet Length Min,Flow Bytes/s,Flow Packets/s,...,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Init_Win_bytes_forward,Init_Win_bytes_backward,Active Mean,Active Std,Active Max,Idle Std
0,54865,3,2,12,6,6,0,0,4.000000e+06,666666.687500,...,0,0,0,0,33,-1,0.0,0.0,0,0.0
1,55054,109,1,6,6,6,6,6,1.100917e+05,18348.623047,...,0,0,0,0,29,256,0.0,0.0,0,0.0
2,55055,52,1,6,6,6,6,6,2.307692e+05,38461.539062,...,0,0,0,0,29,256,0.0,0.0,0,0.0
3,46236,34,1,6,6,6,6,6,3.529412e+05,58823.531250,...,0,0,0,0,31,329,0.0,0.0,0,0.0
4,54863,3,2,12,6,6,0,0,4.000000e+06,666666.687500,...,0,0,0,0,32,-1,0.0,0.0,0,0.0


In [17]:
df.columns

Index([' Destination Port', ' Flow Duration', ' Total Fwd Packets',
       'Total Length of Fwd Packets', ' Fwd Packet Length Max',
       ' Fwd Packet Length Min', 'Bwd Packet Length Max',
       ' Bwd Packet Length Min', 'Flow Bytes/s', ' Flow Packets/s',
       ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Min', 'Bwd IAT Total',
       ' Bwd IAT Std', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags',
       ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length',
       ' Bwd Packets/s', ' Min Packet Length', 'FIN Flag Count',
       ' RST Flag Count', ' PSH Flag Count', ' ACK Flag Count',
       ' URG Flag Count', ' Down/Up Ratio', 'Fwd Avg Bytes/Bulk',
       ' Fwd Avg Packets/Bulk', ' Fwd Avg Bulk Rate', ' Bwd Avg Bytes/Bulk',
       ' Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Init_Win_bytes_forward',
       ' Init_Win_bytes_backward', 'Active Mean', ' Active Std', ' Active Max',
       ' Idle Std', ' Label'],
      dtype='object')

In [18]:
y.head()

0    BENIGN
1    BENIGN
2    BENIGN
3    BENIGN
4    BENIGN
Name:  Label, dtype: object

In [19]:
le = LabelEncoder()
y = le.fit_transform(y)

In [20]:
print(y[:5])

[0 0 0 0 0]


In [21]:
cols = list(X.columns)
for col in cols:
    X[col] = stats.zscore(X[col])

In [22]:
X = X.values.reshape((X.shape[0], X.shape[1], 1))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

In [23]:
inputs = Input(shape=(X.shape[1], 1))
x = Conv1D(32, 3, activation='relu')(inputs)
x = Conv1D(32, 3, activation='relu')(x)
x = Dropout(0.5)(x)
x = MaxPooling1D(2)(x)
x = Conv1D(64, 3, activation='relu')(x)
x = Conv1D(64, 3, activation='relu')(x)
x = Dropout(0.5)(x)
x = MaxPooling1D(2)(x)
x = Conv1D(128, 3, activation='relu')(x)
x = Conv1D(128, 3, activation='relu')(x)
x = Dropout(0.5)(x)
x = MaxPooling1D(2)(x)
x = LSTM(64, return_sequences=True)(x)
x = LSTM(64)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
outputs = Dense(1, activation='sigmoid')(x)


In [24]:
model = Model(inputs=inputs, outputs=outputs)
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 41, 1)]           0         
                                                                 
 conv1d (Conv1D)             (None, 39, 32)            128       
                                                                 
 conv1d_1 (Conv1D)           (None, 37, 32)            3104      
                                                                 
 dropout (Dropout)           (None, 37, 32)            0         
                                                                 
 max_pooling1d (MaxPooling1D  (None, 18, 32)           0         
 )                                                               
                                                                 
 conv1d_2 (Conv1D)           (None, 16, 64)            6208      
                                                             

In [25]:
from keras import backend as K

def precision(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    return precision

def recall(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    return true_positives / (possible_positives + K.epsilon())

def f1_score(y_true, y_pred):
    y_true = K.round(y_true)
    y_pred = K.round(y_pred)
    tp = K.sum(y_true * y_pred)
    fp = K.sum((1 - y_true) * y_pred)
    fn = K.sum(y_true * (1 - y_pred))
    precision = tp / (tp + fp + K.epsilon())
    recall = tp / (tp + fn + K.epsilon())
    f1_score = 2 * precision * recall / (precision + recall)
    return f1_score

In [26]:
adam = Adam(learning_rate=0.001)
model.compile(loss='binary_crossentropy', optimizer=adam, metrics=['accuracy', precision, recall, f1_score])
model.fit(X_train, y_train, epochs=20)

Epoch 1/20
53876/53876 [==============================] - 1196s 207ms/step - loss: 1.2521 - accuracy: 0.8384 - precision: 0.7052 - recall: 0.6861 - F1 score: 0.6953
Epoch 2/20
53876/53876 [==============================] - 1098s 203ms/step - loss: 1.2518 - accuracy: 0.8584 - precision: 0.7172 - recall: 0.7161 - F1 score: 0.7166
Epoch 3/20
53876/53876 [==============================] - 1086s 201ms/step - loss: 1.2234 - accuracy: 0.8788 - precision: 0.7251 - recall: 0.7397 - F1 score: 0.7323
Epoch 4/20
53876/53876 [==============================] - 1021s 200ms/step - loss: 1.1950 - accuracy: 0.8992 - precision: 0.7328 - recall: 0.7666 - F1 score: 0.7492
Epoch 5/20
53876/53876 [==============================] - 1205s 223ms/step - loss: 1.1666 - accuracy: 0.8843 - precision: 0.7441 - recall: 0.7949 - F1 score: 0.7684
Epoch 6/20
53876/53876 [==============================] - 1183s 219ms/step - loss: 1.2521 - accuracy: 0.9196 - precision: 0.7566 - recall: 0.8256 - F1 score: 0.7929
Epoch 7/20

In [27]:
model.save('current_model.h5')
print('Successfully saved the model')

Successfully saved the model


In [32]:
import tensorflow as tf
import pandas as pd
import numpy as np
import os
import time
import h5py
from scapy.all import *
from sklearn.model_selection import train_test_split

In [33]:
last_packet_time = 0
packet_count = 0
active_total = 0
active_mean = 0
active_std = 0
active_max = 0
idle_total = 0
idle_mean = 0
idle_std = 0


fields = [' Destination Port', ' Flow Duration', ' Total Fwd Packets',
       'Total Length of Fwd Packets', ' Fwd Packet Length Max',
       ' Fwd Packet Length Min', 'Bwd Packet Length Max',
       ' Bwd Packet Length Min', 'Flow Bytes/s', ' Flow Packets/s',
       ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Min', 'Bwd IAT Total',
       ' Bwd IAT Std', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags',
       ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length',
       ' Bwd Packets/s', ' Min Packet Length', 'FIN Flag Count',
       ' RST Flag Count', ' PSH Flag Count', ' ACK Flag Count',
       ' URG Flag Count', ' Down/Up Ratio', 'Fwd Avg Bytes/Bulk',
       ' Fwd Avg Packets/Bulk', ' Fwd Avg Bulk Rate', ' Bwd Avg Bytes/Bulk',
       ' Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Init_Win_bytes_forward',
       ' Init_Win_bytes_backward', 'Active Mean', ' Active Std', ' Active Max',
       ' Idle Std']

packet_data = []
src_ips = []

def capture_packet(packet):
    global packet_count, active_total, active_mean, active_std, active_max, idle_std, idle_total, idle_mean, last_packet_time
    packet_count += 1
    try:
        if IP in packet:
            destination_port, flow_duration, total_fwd_packets, total_length_fwd_packets, fwd_packet_length_max, fwd_packet_length_min, bwd_packet_length_max, bwd_packet_length_min, flow_bytes_per_sec, flow_packets_per_sec, fwd_psh_flags,bwd_psh_flags, fwd_urg_flags, bwd_urg_flags, fwd_header_length, bwd_header_length, bwd_packets_per_sec, min_packet_length, fin_flag_count, rst_flag_count, psh_flag_count, ack_flag_count, urg_flag_count, down_up_ratio, fwd_avg_bytes_bulk, fwd_avg_packets_bulk, fwd_avg_bulk_rate, bwd_avg_bytes_bulk, bwd_avg_packets_bulk, bwd_avg_bulk_rate, init_win_bytes_forward, init_win_bytes_backward=0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
            bwd_iat=0
            destination_port = packet[IP].dport
            src_ip = packet[IP].src
            dst_ip = packet[IP].dst
            protocol = packet[IP].proto
            packet_length = packet[IP].len
            flow_iat = packet.time - last_packet_time

            if TCP in packet:
                source_ip = packet[IP].src
                src_port = packet[TCP].sport
                dst_port = packet[TCP].dport
                flow_iat = packet.time - last_packet_time
                bwd_iat = packet.time - last_packet_time - flow_iat
                # Extract TCP layer attributes
                flow_duration = packet.time - last_packet_time
                total_fwd_packets = packet[TCP].flags & 0x1
                total_length_fwd_packets = packet[TCP].options[0][1]
                fwd_packet_length_max = packet[TCP].window
                fwd_packet_length_min = packet[TCP].options[0][1]
                bwd_packet_length_max = packet[TCP].ack
                bwd_packet_length_min = packet[TCP].seq
                flow_bytes_per_sec = packet[IP].frag
                flow_packets_per_sec = 1 / packet.time_delta
                fwd_psh_flags = packet[TCP].flags.PSH
                bwd_psh_flags = packet[TCP].flags.PSH
                fwd_urg_flags = packet[TCP].flags.URG
                bwd_urg_flags = packet[TCP].flags.URG
                fwd_header_length = packet[TCP].dataofs * 4
                bwd_header_length = packet[TCP].dataofs * 4
                bwd_packets_per_sec = 1 / packet.time_delta
                min_packet_length = packet[TCP].options[0][1]
                fin_flag_count = packet[TCP].flags.FIN
                rst_flag_count = packet[TCP].flags.RST
                psh_flag_count = packet[TCP].flags.PSH
                ack_flag_count = packet[TCP].flags.ACK
                urg_flag_count = packet[TCP].flags.URG
                down_up_ratio = packet[IP].len / packet[IP].id
                fwd_avg_bytes_bulk = packet[TCP].options[0][1]
                fwd_avg_packets_bulk = packet[TCP].flags & 0x1
                fwd_avg_bulk_rate = packet[TCP].ack / packet[TCP].seq
                bwd_avg_bytes_bulk = packet[TCP].options[0][1]
                bwd_avg_packets_bulk = packet[TCP].flags & 0x1
                bwd_avg_bulk_rate = packet[TCP].ack / packet[TCP].seq
                init_win_bytes_forward = packet[TCP].options[0][1]
                init_win_bytes_backward = packet[TCP].seq
            
                if packet_count > 1:
                    active_time = packet.time - last_packet_time - flow_iat
                    active_total += active_time
                    active_mean = active_total / packet_count
                    active_std += (active_time - active_mean) ** 2

                    idle_time = flow_iat - active_time
                    idle_total += idle_time
                    idle_mean = idle_total / packet_count
                    idle_std += (idle_time - idle_mean) ** 2
            

            packet_data.append([destination_port, flow_duration, total_fwd_packets, total_length_fwd_packets, fwd_packet_length_max, fwd_packet_length_min, bwd_packet_length_max, bwd_packet_length_min, flow_bytes_per_sec, flow_packets_per_sec, flow_iat, flow_iat, flow_iat, bwd_iat, bwd_iat, fwd_psh_flags,bwd_psh_flags, fwd_urg_flags, bwd_urg_flags, fwd_header_length, bwd_header_length, bwd_packets_per_sec, min_packet_length, fin_flag_count, rst_flag_count, psh_flag_count, ack_flag_count, urg_flag_count, down_up_ratio, fwd_avg_bytes_bulk, fwd_avg_packets_bulk, fwd_avg_bulk_rate, bwd_avg_bytes_bulk, bwd_avg_packets_bulk, bwd_avg_bulk_rate, init_win_bytes_forward, init_win_bytes_backward, active_mean, active_std, active_max, idle_std])
            src_ips.append(source_ip)
    except Exception as e:
        print("Ignoring the non-TCP packets")

try:
    sniff(prn=capture_packet, count=100)

    df_rtd = pd.DataFrame(packet_data, columns=fields)

    df_rtd.to_csv(r'C:\Users\mahes\Downloads\MachineLearningCSV\MachineLearningCVE\captured_packets.csv', index=False)

    print("Packet capture completed successfully.")
except Exception as e:
    print()

Ignoring the non-TCP packets
Ignoring the non-TCP packets
Ignoring the non-TCP packets
Ignoring the non-TCP packets
Ignoring the non-TCP packets
Ignoring the non-TCP packets
Ignoring the non-TCP packets
Packet capture completed successfully.


In [34]:
df = pd.read_csv(r'C:\Users\mahes\Downloads\MachineLearningCSV\MachineLearningCVE\captured_packets.csv')
display(df)

,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Bwd Packet Length Max,Bwd Packet Length Min,Flow Bytes/s,Flow Packets/s,...,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Init_Win_bytes_forward,Init_Win_bytes_backward,Active Mean,Active Std,Active Max,Idle Std
0,1947,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,137,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,138,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,137,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,57621,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [35]:
from tensorflow.keras.models import load_model
df = pd.read_csv(r'C:\Users\mahes\Downloads\MachineLearningCSV\MachineLearningCVE\captured_packets.csv')
tf.get_logger().setLevel('ERROR')

if len(df) == 0:
    print("Scapy did not capture any TCP Packets. Please try again!")

else:
    model = load_model(r'C:\Users\mahes\Downloads\MachineLearningCSV\MachineLearningCVE\current_model.h5')
    predictions = model.predict(df)
    predictions = np.where(predictions < 0.5, 0, 1)  # Set values less than 0.5 to 0, and others to 1
    prediction_list = predictions.flatten()
    print(prediction_list)

1/1 [==============================] - 2s 2s/step
[0 0 0 0 0]


In [36]:
for i in range(0, len(prediction_list)-1):
    if(prediction_list[i] == 1):
        print("The captured packet is malicious!!!\n Please press 1 to block all the packets from the source IP: " + src_ips[i])
        choice = input()
        if(choice == 1):
            subprocess.run(["iptables", "-A", "INPUT", "-s", src_ips[i], "-j", "DROP"])
            print("Successfully blocked all the packets receiving from " + src_ips[i])
        else:
            print("Ignored the trusted IP.")